# SpatialMesh — Day 7: Graph Builder
**Goal:** Take 4 speaker mono audio streams → build a directed weighted PyTorch Geometric graph ready for GAT-GNN

**Output:** `SpatialMeshGraph` — a PyG `Data` object with:
- `x` → node features [4, 133]
- `edge_index` → directed edges [2, 12]
- `edge_attr` → edge features [12, 7]
- `activity_mask` → who is speaking [4]
- `dominance` → conversational dominance [4]
- `overlap_duration` → pairwise overlap [4, 4]
- `prev_positions` → last frame positions [4, 2]

In [ ]:
# Cell 1 — Install PyTorch Geometric
import subprocess, sys

def install_pyg():
    import torch
    version = torch.__version__.split('+')[0]
    cuda = 'cu118' if torch.cuda.is_available() else 'cpu'
    print(f'PyTorch: {version} | Device: {cuda}')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch-geometric',
        'torch-scatter', 'torch-sparse',
        '-f', f'https://data.pyg.org/whl/torch-{version}+{cuda}.html'
    ])
    print('PyG installed.')

install_pyg()

In [ ]:
# Cell 2 — Imports
import torch
import torch.nn.functional as F
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from itertools import permutations
from torch_geometric.data import Data
from google.colab import drive

drive.mount('/content/drive')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Constants — locked from architecture decision
N_SPEAKERS      = 4
CNN_DIM         = 128
NODE_DIM        = 133   # 128 + rms + dominance + activity + az + el
EDGE_DIM        = 7     # per directed edge
N_EDGES         = 12    # 4 speakers, fully directed
SR              = 48000
SEGMENT_SEC     = 3.0   # seconds per audio chunk
OVERLAP_WINDOW  = 5.0   # seconds for dominance computation
ACTIVITY_THRESH = 0.01  # RMS threshold for activity detection
ALPHA_SIGMOID   = 5.0   # sigmoid sharpness for dominance

print('Imports done.')

In [ ]:
# Cell 3 — Load CNN Encoder
import torch.nn as nn

class CNNEncoder(nn.Module):
    """
    Exact architecture from Day 5 training.
    4-layer 1D CNN, stereo input (2ch), 128-dim output embedding.
    """
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            # Layer 1
            nn.Conv1d(2, 32, kernel_size=64, stride=4, padding=30),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(4),

            # Layer 2
            nn.Conv1d(32, 64, kernel_size=32, stride=2, padding=15),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(4),

            # Layer 3
            nn.Conv1d(64, 128, kernel_size=16, stride=2, padding=7),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(4),

            # Layer 4
            nn.Conv1d(128, 128, kernel_size=8, stride=2, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x):
        # x: [batch, 2, samples]
        out = self.encoder(x)   # [batch, 128, 1]
        return out.squeeze(-1)  # [batch, 128]


# Load saved weights
CNN_PATH = '/content/drive/MyDrive/SpatialMesh/models/cnn_encoder_best.pt'

cnn_encoder = CNNEncoder().to(DEVICE)
cnn_encoder.load_state_dict(torch.load(CNN_PATH, map_location=DEVICE))
cnn_encoder.eval()

print(f'CNN encoder loaded from {CNN_PATH}')
print(f'Parameters: {sum(p.numel() for p in cnn_encoder.parameters()):,}')

In [ ]:
# Cell 4 — Load 4 LibriSpeech Speakers
# Expects mono .flac files — one per speaker
# Adjust paths to your LibriSpeech structure

LIBRISPEECH_PATH = '/content/drive/MyDrive/SpatialMesh/data/librispeech/'

def load_speaker_audio(flac_path, sr=SR, duration=SEGMENT_SEC):
    """
    Load a mono audio segment from a LibriSpeech .flac file.
    Returns numpy array [samples] at target SR.
    """
    audio, orig_sr = sf.read(flac_path)
    if audio.ndim > 1:
        audio = audio[:, 0]  # take left channel if stereo
    if orig_sr != sr:
        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=sr)
    n_samples = int(duration * sr)
    if len(audio) >= n_samples:
        audio = audio[:n_samples]
    else:
        audio = np.pad(audio, (0, n_samples - len(audio)))
    return audio.astype(np.float32)


# Find 4 random LibriSpeech files
import random
all_flacs = list(Path(LIBRISPEECH_PATH).rglob('*.flac'))
random.seed(42)
selected = random.sample(all_flacs, N_SPEAKERS)

raw_audios = []
for i, path in enumerate(selected):
    audio = load_speaker_audio(path)
    raw_audios.append(audio)
    print(f'Speaker {i}: {path.name} | shape: {audio.shape} | '
          f'RMS: {np.sqrt(np.mean(audio**2)):.4f}')

print(f'\nLoaded {len(raw_audios)} speakers, {SEGMENT_SEC}s each at {SR}Hz')

In [ ]:
# Cell 5 — Conversational Feature Extraction
# Computes per-speaker features needed for node construction

def compute_conversational_features(audios, sr=SR,
                                     activity_thresh=ACTIVITY_THRESH):
    """
    For each speaker compute:
    - rms_energy: current loudness normalized [0,1]
    - activity_flag: 1 if speaking, 0 if silent
    - dominance_score: fraction of segment this speaker is active
    - overlap_duration [N,N]: seconds speakers i and j overlap
    """
    N = len(audios)

    # Frame-level activity detection
    # Use 20ms frames — standard for speech activity
    frame_len = int(0.02 * sr)   # 20ms
    hop_len   = int(0.01 * sr)   # 10ms hop

    # Per-speaker frame-level RMS
    frame_activity = []  # [N, n_frames] binary
    rms_values = []

    for audio in audios:
        # Compute RMS per frame
        frames = librosa.util.frame(
            audio, frame_length=frame_len, hop_length=hop_len
        )
        rms_per_frame = np.sqrt(np.mean(frames**2, axis=0))
        active_frames = (rms_per_frame > activity_thresh).astype(float)
        frame_activity.append(active_frames)
        rms_values.append(np.sqrt(np.mean(audio**2)))

    frame_activity = np.array(frame_activity)  # [N, n_frames]
    rms_values     = np.array(rms_values)      # [N]

    # Normalize RMS to [0,1]
    rms_max = rms_values.max() + 1e-8
    rms_norm = rms_values / rms_max

    # Activity flag — is speaker active in this segment?
    activity_flag = (rms_norm > activity_thresh).astype(float)

    # Dominance score — fraction of frames speaker is active
    dominance_score = frame_activity.mean(axis=1)  # [N]
    dom_max = dominance_score.max() + 1e-8
    dominance_norm = dominance_score / dom_max

    # Pairwise overlap duration [N, N]
    # overlap(i,j) = seconds where BOTH i and j are active
    n_frames = frame_activity.shape[1]
    frame_duration = hop_len / sr  # seconds per frame
    overlap_duration = np.zeros((N, N))

    for i in range(N):
        for j in range(N):
            if i != j:
                both_active = frame_activity[i] * frame_activity[j]
                overlap_duration[i][j] = both_active.sum() * frame_duration

    return {
        'rms_energy':       rms_norm,           # [N] float
        'activity_flag':    activity_flag,       # [N] binary
        'dominance_score':  dominance_norm,      # [N] float
        'overlap_duration': overlap_duration,    # [N,N] seconds
        'frame_activity':   frame_activity,      # [N, n_frames]
    }


conv_features = compute_conversational_features(raw_audios)

print('Conversational features:')
print(f'  rms_energy:      {conv_features["rms_energy"]}')
print(f'  activity_flag:   {conv_features["activity_flag"]}')
print(f'  dominance_score: {conv_features["dominance_score"]}')
print(f'  overlap_duration:\n{conv_features["overlap_duration"].round(3)}')

In [ ]:
# Cell 6 — Extract CNN Embeddings
# Run each speaker audio through the trained CNN encoder
# Input must be stereo (2ch) — replicate mono to stereo

def extract_cnn_embeddings(audios, encoder, device=DEVICE, sr=SR):
    """
    For each mono audio:
    1. Replicate to stereo [2, samples] 
       (CNN was trained on HRTF-convolved stereo)
    2. Pass through CNN encoder
    3. Return [N, 128] embedding matrix
    
    NOTE: In the full pipeline, audio would be HRTF-convolved first.
    For Day 7 graph builder testing, we replicate mono→stereo
    as a placeholder. Day 12 integration will use real HRTF convolution.
    """
    embeddings = []

    with torch.no_grad():
        for audio in audios:
            # Replicate mono to stereo: [samples] → [2, samples]
            stereo = np.stack([audio, audio], axis=0)
            tensor = torch.tensor(stereo).unsqueeze(0).to(device)
            # tensor: [1, 2, samples]

            emb = encoder(tensor)  # [1, 128]
            embeddings.append(emb.squeeze(0).cpu())

    return torch.stack(embeddings)  # [N, 128]


embeddings = extract_cnn_embeddings(raw_audios, cnn_encoder)

print(f'CNN embeddings shape: {embeddings.shape}')  # [4, 128]
print(f'Embedding norms: {embeddings.norm(dim=1).numpy().round(4)}')

# Quick sanity — embeddings should be different per speaker
for i in range(N_SPEAKERS):
    for j in range(i+1, N_SPEAKERS):
        sim = F.cosine_similarity(
            embeddings[i].unsqueeze(0),
            embeddings[j].unsqueeze(0)
        ).item()
        print(f'  Cosine similarity Speaker {i} vs {j}: {sim:.4f}')

In [ ]:
# Cell 7 — Build Node Features [4, 133]

def build_node_features(embeddings, conv_features,
                         current_positions=None):
    """
    Assemble 133-dim node feature vector per speaker.

    Structure:
    [0:128]   CNN embedding          — spatial voice identity
    [128]     rms_energy             — current loudness
    [129]     dominance_score        — conversational dominance
    [130]     activity_flag          — currently speaking
    [131]     current_azimuth        — normalized [-1, 1]
    [132]     current_elevation      — normalized [-1, 1]

    Args:
        embeddings:        [N, 128] CNN embeddings
        conv_features:     dict from compute_conversational_features
        current_positions: [N, 2] current (az, el) normalized [-1,1]
                           None → initialize to spread positions
    """
    N = embeddings.shape[0]

    # Initialize positions if first frame
    # Spread evenly: 0°, 90°, 180°, 270° → normalized
    if current_positions is None:
        init_az = [0.0, 0.5, 1.0, -0.5]   # 0°, 90°, 180°, -90°
        init_el = [0.0, 0.0, 0.0,  0.0]   # all at horizontal plane
        current_positions = torch.tensor(
            list(zip(init_az, init_el)), dtype=torch.float32
        )  # [N, 2]

    # Assemble node features
    rms       = torch.tensor(conv_features['rms_energy'],
                              dtype=torch.float32).unsqueeze(1)      # [N,1]
    dominance = torch.tensor(conv_features['dominance_score'],
                              dtype=torch.float32).unsqueeze(1)      # [N,1]
    activity  = torch.tensor(conv_features['activity_flag'],
                              dtype=torch.float32).unsqueeze(1)      # [N,1]
    az        = current_positions[:, 0].unsqueeze(1)                  # [N,1]
    el        = current_positions[:, 1].unsqueeze(1)                  # [N,1]

    node_features = torch.cat([
        embeddings,   # [N, 128]
        rms,          # [N, 1]
        dominance,    # [N, 1]
        activity,     # [N, 1]
        az,           # [N, 1]
        el,           # [N, 1]
    ], dim=1)         # [N, 133]

    return node_features, current_positions


node_features, init_positions = build_node_features(
    embeddings, conv_features
)

print(f'Node features shape: {node_features.shape}')  # [4, 133]
print(f'Initial positions (az, el normalized):\n{init_positions}')
print(f'\nNode feature breakdown:')
print(f'  CNN embedding [0:128]: mean={node_features[:, :128].mean():.4f}')
print(f'  RMS energy    [128]:   {node_features[:, 128].numpy().round(4)}')
print(f'  Dominance     [129]:   {node_features[:, 129].numpy().round(4)}')
print(f'  Activity flag [130]:   {node_features[:, 130].numpy()}')
print(f'  Azimuth       [131]:   {node_features[:, 131].numpy().round(4)}')
print(f'  Elevation     [132]:   {node_features[:, 132].numpy().round(4)}')

In [ ]:
# Cell 8 — Build Directed Edge Index [2, 12]

def build_edge_index(n_speakers=N_SPEAKERS):
    """
    Fully connected directed graph.
    Every speaker i has a directed edge to every speaker j (i ≠ j).
    4 speakers → 4 × 3 = 12 directed edges.

    Edge (i→j) semantics: how much does speaker i interfere with speaker j?
    Directed because dominance(A→B) ≠ dominance(B→A)

    Returns:
        edge_index: [2, 12] LongTensor
        edge_pairs: list of (i, j) tuples for feature computation
    """
    sources = []
    targets = []
    pairs   = []

    for i in range(n_speakers):
        for j in range(n_speakers):
            if i != j:
                sources.append(i)
                targets.append(j)
                pairs.append((i, j))

    edge_index = torch.tensor([sources, targets], dtype=torch.long)
    return edge_index, pairs


edge_index, edge_pairs = build_edge_index()

print(f'Edge index shape: {edge_index.shape}')  # [2, 12]
print(f'Directed edges:')
for k, (i, j) in enumerate(edge_pairs):
    print(f'  Edge {k:2d}: Speaker {i} → Speaker {j}')

In [ ]:
# Cell 9 — Compute Edge Features [12, 7]

def compute_edge_features(embeddings, conv_features,
                            current_positions, edge_pairs):
    """
    Compute 7-dim feature vector for each directed edge (i→j).

    Edge features — locked architecture:
    [0] delta_azimuth       — az_i - az_j, signed [-2, 2]
    [1] delta_elevation     — el_i - el_j, signed [-2, 2]
    [2] spectral_correlation— cosine(embed_i, embed_j) [-1, 1]
                              perceptual masking risk
    [3] relative_db         — RMS_i - RMS_j, normalized [-1, 1]
                              instantaneous dominance direction
    [4] dominance_ratio     — sigmoid(alpha*(dom_i - dom_j)) [0, 1]
                              temporal directional dominance
    [5] overlap_flag        — 1 if both active simultaneously
    [6] overlap_duration    — seconds of co-occurrence, normalized [0,1]
    """
    rms       = torch.tensor(conv_features['rms_energy'],    dtype=torch.float32)
    dominance = torch.tensor(conv_features['dominance_score'],dtype=torch.float32)
    activity  = torch.tensor(conv_features['activity_flag'], dtype=torch.float32)
    overlap   = torch.tensor(conv_features['overlap_duration'],dtype=torch.float32)

    # Normalize overlap duration to [0,1] — max 5 seconds
    overlap_norm = torch.clamp(overlap / OVERLAP_WINDOW, 0.0, 1.0)

    edge_features = []

    for (i, j) in edge_pairs:

        # [0] delta_azimuth — signed geometric separation
        delta_az = (current_positions[i, 0]
                    - current_positions[j, 0]).item()

        # [1] delta_elevation — signed geometric separation
        delta_el = (current_positions[i, 1]
                    - current_positions[j, 1]).item()

        # [2] spectral_correlation — perceptual masking risk
        # cosine similarity of CNN embeddings
        spec_corr = F.cosine_similarity(
            embeddings[i].unsqueeze(0),
            embeddings[j].unsqueeze(0)
        ).item()

        # [3] relative_db — instantaneous dominance direction
        # RMS difference normalized to [-1, 1]
        rel_db = (rms[i] - rms[j]).item()
        rel_db = float(np.clip(rel_db, -1.0, 1.0))

        # [4] dominance_ratio — temporal directional dominance
        # sigmoid gives clean asymmetric signal
        dom_diff  = ALPHA_SIGMOID * (dominance[i] - dominance[j])
        dom_ratio = torch.sigmoid(dom_diff).item()

        # [5] overlap_flag — are BOTH speakers currently active?
        ovlp_flag = float(
            activity[i].item() == 1.0 and activity[j].item() == 1.0
        )

        # [6] overlap_duration — normalized seconds of co-occurrence
        ovlp_dur = overlap_norm[i][j].item()

        edge_features.append([
            delta_az,    # [0]
            delta_el,    # [1]
            spec_corr,   # [2]
            rel_db,      # [3]
            dom_ratio,   # [4]
            ovlp_flag,   # [5]
            ovlp_dur,    # [6]
        ])

    return torch.tensor(edge_features, dtype=torch.float32)  # [12, 7]


edge_attr = compute_edge_features(
    embeddings, conv_features, init_positions, edge_pairs
)

print(f'Edge features shape: {edge_attr.shape}')  # [12, 7]
print(f'\nEdge feature matrix (12 edges × 7 features):')
print(f'{"Edge":>6} {"dAz":>7} {"dEl":>7} {"SpCorr":>7}',
      f'{"RelDB":>7} {"DomRat":>7} {"OvFlag":>7} {"OvDur":>7}')
for k, ((i,j), feat) in enumerate(zip(edge_pairs, edge_attr)):
    print(f'  {i}→{j}  ',
          '  '.join(f'{v.item():7.4f}' for v in feat))

In [ ]:
# Cell 10 — Assemble PyTorch Geometric Data Object

def build_spatialmesh_graph(raw_audios, cnn_encoder,
                             prev_positions=None, device=DEVICE):
    """
    Full graph builder — takes 4 raw mono audio streams,
    returns a PyG Data object ready for GAT-GNN.

    Args:
        raw_audios:    list of 4 numpy arrays [samples]
        cnn_encoder:   trained CNNEncoder
        prev_positions:[4, 2] normalized positions from last frame
                       None → initialize to spread
        device:        torch device

    Returns:
        graph: PyG Data object with all fields populated
    """
    # Step 1 — Conversational feature extraction
    conv_features = compute_conversational_features(raw_audios)

    # Step 2 — CNN embeddings
    embeddings = extract_cnn_embeddings(raw_audios, cnn_encoder, device)

    # Step 3 — Node features [4, 133]
    node_features, current_positions = build_node_features(
        embeddings, conv_features, prev_positions
    )

    # Step 4 — Edge index [2, 12]
    edge_index, edge_pairs = build_edge_index()

    # Step 5 — Edge features [12, 7]
    edge_attr = compute_edge_features(
        embeddings, conv_features, current_positions, edge_pairs
    )

    # Step 6 — Assemble PyG Data
    graph = Data(
        x           = node_features,    # [4, 133]
        edge_index  = edge_index,        # [2, 12]
        edge_attr   = edge_attr,         # [12, 7]
    )

    # Attach extra tensors needed by loss function
    graph.activity_mask    = torch.tensor(
        conv_features['activity_flag'], dtype=torch.float32
    )  # [4]

    graph.dominance        = torch.tensor(
        conv_features['dominance_score'], dtype=torch.float32
    )  # [4]

    graph.overlap_duration = torch.tensor(
        conv_features['overlap_duration'], dtype=torch.float32
    )  # [4, 4]

    graph.prev_positions   = current_positions  # [4, 2]

    graph.embeddings       = embeddings         # [4, 128] for loss

    return graph


# Build the graph
graph = build_spatialmesh_graph(raw_audios, cnn_encoder)

print('=== SpatialMesh Graph ===')
print(f'x (node features):      {graph.x.shape}')           # [4, 133]
print(f'edge_index:             {graph.edge_index.shape}')  # [2, 12]
print(f'edge_attr:              {graph.edge_attr.shape}')   # [12, 7]
print(f'activity_mask:          {graph.activity_mask}')     # [4]
print(f'dominance:              {graph.dominance}')         # [4]
print(f'overlap_duration shape: {graph.overlap_duration.shape}')  # [4,4]
print(f'prev_positions:         {graph.prev_positions}')   # [4, 2]
print(f'embeddings:             {graph.embeddings.shape}') # [4, 128]
print(f'\nTotal node features:   {graph.x.shape[1]}')      # 133
print(f'Total edge features:   {graph.edge_attr.shape[1]}') # 7
print(f'\nGraph is valid PyG Data: {graph.validate()}')

In [ ]:
# Cell 11 — Validate Graph Structure
# Quick sanity checks before moving to Day 8

def validate_graph(graph):
    errors = []
    warnings = []

    # Shape checks
    assert graph.x.shape          == (4, 133), \
        f'Node features wrong shape: {graph.x.shape}'
    assert graph.edge_index.shape == (2, 12),  \
        f'Edge index wrong shape: {graph.edge_index.shape}'
    assert graph.edge_attr.shape  == (12, 7),  \
        f'Edge attr wrong shape: {graph.edge_attr.shape}'

    # No NaN or Inf
    for name, tensor in [
        ('x', graph.x),
        ('edge_attr', graph.edge_attr),
        ('prev_positions', graph.prev_positions),
    ]:
        if torch.isnan(tensor).any():
            errors.append(f'NaN found in {name}')
        if torch.isinf(tensor).any():
            errors.append(f'Inf found in {name}')

    # Edge connectivity — all nodes should have edges
    for node in range(4):
        has_out = (graph.edge_index[0] == node).any()
        has_in  = (graph.edge_index[1] == node).any()
        if not has_out or not has_in:
            errors.append(f'Node {node} missing edges')

    # Spectral correlation range check
    spec_corr = graph.edge_attr[:, 2]
    if spec_corr.min() < -1.0 or spec_corr.max() > 1.0:
        warnings.append(
            f'Spectral correlation out of [-1,1]: '
            f'min={spec_corr.min():.4f} max={spec_corr.max():.4f}'
        )

    # Dominance ratio range check [0,1]
    dom_ratio = graph.edge_attr[:, 4]
    if dom_ratio.min() < 0.0 or dom_ratio.max() > 1.0:
        warnings.append(
            f'Dominance ratio out of [0,1]: '
            f'min={dom_ratio.min():.4f} max={dom_ratio.max():.4f}'
        )

    # Activity mask sanity
    n_active = int(graph.activity_mask.sum().item())
    if n_active == 0:
        warnings.append('All speakers silent — check activity threshold')

    # Report
    print('=== Graph Validation ===')
    if not errors and not warnings:
        print('✅ All checks passed')
    for e in errors:
        print(f'❌ ERROR:   {e}')
    for w in warnings:
        print(f'⚠️  WARNING: {w}')

    print(f'\nActive speakers: {n_active}/4')
    print(f'Spectral correlations (masking risk):')
    for k, (i, j) in enumerate(edge_pairs):
        sc = graph.edge_attr[k, 2].item()
        risk = 'HIGH' if sc > 0.7 else 'MED' if sc > 0.4 else 'LOW'
        print(f'  Speaker {i}→{j}: {sc:.4f} [{risk}]')

    return len(errors) == 0


passed = validate_graph(graph)
print(f'\nDay 7 status: {"✅ DONE" if passed else "❌ FIX ERRORS ABOVE"}')

In [ ]:
# Cell 12 — Save Graph Builder as Importable Module
# Other notebooks (Day 8, 9, 12) will import this

import json

# Save one example graph for Day 8 architecture testing
SAVE_PATH = '/content/drive/MyDrive/SpatialMesh/graphs/'
Path(SAVE_PATH).mkdir(parents=True, exist_ok=True)

torch.save({
    'x':               graph.x,
    'edge_index':      graph.edge_index,
    'edge_attr':       graph.edge_attr,
    'activity_mask':   graph.activity_mask,
    'dominance':       graph.dominance,
    'overlap_duration':graph.overlap_duration,
    'prev_positions':  graph.prev_positions,
    'embeddings':      graph.embeddings,
}, SAVE_PATH + 'example_graph.pt')

# Save architecture constants for downstream notebooks
constants = {
    'NODE_DIM':        NODE_DIM,
    'EDGE_DIM':        EDGE_DIM,
    'N_SPEAKERS':      N_SPEAKERS,
    'N_EDGES':         N_EDGES,
    'SR':              SR,
    'SEGMENT_SEC':     SEGMENT_SEC,
    'OVERLAP_WINDOW':  OVERLAP_WINDOW,
    'ACTIVITY_THRESH': ACTIVITY_THRESH,
    'ALPHA_SIGMOID':   ALPHA_SIGMOID,
}
with open(SAVE_PATH + 'graph_constants.json', 'w') as f:
    json.dump(constants, f, indent=2)

print(f'Example graph saved to {SAVE_PATH}example_graph.pt')
print(f'Constants saved to {SAVE_PATH}graph_constants.json')
print('\n=== Day 7 Complete ===')
print(f'Graph builder ready for Day 8 GAT-GNN architecture.')
print(f'\nNode features:  {NODE_DIM}-dim')
print(f'Edge features:  {EDGE_DIM}-dim')
print(f'Edges:          {N_EDGES} directed')
print(f'Speakers:       {N_SPEAKERS}')